# BBM418 / AIN433 — Fundamentals of Computer Vision  
## Programming Assignment 2: Object Localization and Single-Object Tracking

### Student Notebook

This notebook is the starter template for the assignment. This is a draft.
You can follow the given structure or you can implement your version.
Feel free to change given code templates.

The assignment has five main parts:

1. Dataset loading and visualization  
2. Search-region sample generation  
3. CNN-based bounding-box regression model  
4. Single-object tracking on validation and test sequences  
5. Validation evaluation and analysis  

Keep your implementation clean, reproducible, and easy to follow.

### Student Info

#### Name:

#### ID:

## Assignment Setting

You will work with the given single-object tracking dataset.

### Training and Validation Sequences

These contain full ground-truth annotations:

```text
SequenceName/
  img/
    0001.jpg
    0002.jpg
    ...
  groundtruth_rect.txt
```

### Test Sequences

These contain only the first-frame initialization box:

```text
SequenceName/
  img/
    0001.jpg
    0002.jpg
    ...
  init_rect.txt
```

## Important Rule

For validation and test tracking, you may use only the first-frame bounding box to initialize the tracker.

After the first frame, all bounding boxes must be predicted by your tracker.

For validation sequences, later ground-truth boxes may be used **only after tracking** to compute evaluation metrics.

For test sequences, ground truth is hidden and must not be used.

## Expected Output

Your notebook should generate prediction CSV files for the test sequences.

Each file must be saved as:

```text
outputs/test/<sequence_name>_predictions.csv
```

The required CSV format is:

```text
frame,x,y,w,h
1,198,214,34,81
2,197.2,214.1,34.5,80.7
3,195.4,213.9,34.8,81.2
```

Coordinates must be full-frame pixel coordinates in `(x, y, width, height)` format. Do **not** save normalized crop coordinates.

# 0. Imports and Configuration

Run this section first. You may add extra imports if needed.

Make sure the paths below match your local folder structure.

In [ ]:
import os
import csv
import math
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms

In Google Colab environment, to download the dataset and extract it to the working directory you may use the following cell.

In [ ]:
# 1. Configuration
FILE_ID = '1q8AAAQ64QwA16gU5jEKDvPure7CVajUt'
ZIP_NAME = 'dataset.zip'

# 2. Download the file
!gdown --id {FILE_ID} -O {ZIP_NAME}

# 3. Extract directly to /content
# This will result in: /content/data/sequences/ and /content/data/splits/
!unzip -q {ZIP_NAME} -d /content/

# 4. Cleanup the zip file
!rm {ZIP_NAME}

# 5. Verify the structure
print("Files in /content/data:", os.listdir("/content/data"))

## Reproducibility

Set random seeds so your results are more reproducible.
Include your seeds if you introduce additional randomness.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Paths and Hyperparameters

Update `DATA_ROOT` if your dataset is located somewhere else.

The suggested hyperparameters are only starting points. You may change them, but report your final choices in the discussion section.

In [ ]:
DATA_ROOT = Path("data/sequences")  # TODO: Change this if necessary
SPLIT_ROOT = Path("data/splits")
OUTPUT_ROOT = Path("outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 224
SEARCH_SCALE = 2.5
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
NUM_WORKERS = 0

print("Device:", DEVICE)
print("DATA_ROOT:", DATA_ROOT)

# PART 1 — Dataset Loading and Visualization

In this part, you will implement utilities for loading the dataset and visualizing bounding boxes.

You need separate loading functions for:

- training/validation sequences with `groundtruth_rect.txt`,
- test sequences with `init_rect.txt`.

This distinction is important because test sequences do not contain full ground truth.

## 1.1 Bounding-Box File Reader

The annotation files contain bounding boxes in the format:

```text
x,y,width,height
```

Some files may use commas, spaces, or tabs as separators. Your parser should handle these cases.

In [ ]:
def read_bbox_file(path: Path) -> np.ndarray:
    """
    Read a bounding-box file.

    The file may contain one line or multiple lines.
    Each line has:
        x, y, width, height

    The parser should handle comma-separated, tab-separated,
    and whitespace-separated values.

    Returns:
        bboxes: numpy array of shape (N, 4)
    """
    # TODO: Implement this function
    raise NotImplementedError

## 1.2 Frame Path Reader

This function should read all frame paths from the `img/` folder and sort them in frame order.

In [ ]:
def get_frame_paths(sequence_path: Path) -> List[Path]:
    """
    Return sorted frame paths from sequence_path/img/.

    Returns:
        frame_paths: sorted list of image paths
    """
    # TODO: Implement this function
    raise NotImplementedError

## 1.3 Load Training / Validation Sequences

Training and validation sequences contain full ground truth.

Your loader must check that:

```text
number of frames == number of ground-truth boxes
```

If there is a mismatch, raise an error. Do not silently truncate frames or annotations.

In [ ]:
def load_sequence_with_gt(sequence_path: Path) -> Tuple[List[Path], np.ndarray]:
    """
    Load a training or validation sequence.

    Expected files:
        img/
        groundtruth_rect.txt

    Returns:
        frame_paths: sorted list of image paths
        bboxes: numpy array of shape (N, 4)

    Required check:
        len(frame_paths) must equal len(bboxes).
    """
    # TODO: Implement this function
    raise NotImplementedError

## 1.4 Load Test Sequences

Test sequences contain only `init_rect.txt`.

This file contains one bounding box: the first-frame initialization box.

Your test loader should not expect `groundtruth_rect.txt`.

In [ ]:
def load_test_sequence(sequence_path: Path) -> Tuple[List[Path], np.ndarray]:
    """
    Load a hidden test sequence.

    Expected files:
        img/
        init_rect.txt

    Returns:
        frame_paths: sorted list of image paths
        initial_bbox: numpy array of shape (4,)
    """
    # TODO: Implement this function
    raise NotImplementedError

## 1.5 Image Reading and Drawing Utilities

Implement helper functions for reading RGB images and drawing bounding boxes.

These functions will be used throughout the notebook.

In [ ]:
def read_image_rgb(path: Path) -> np.ndarray:
    """
    Read an image using OpenCV and convert it from BGR to RGB.
    """
    # TODO: Implement this function
    raise NotImplementedError


def draw_bbox(
    image: np.ndarray,
    bbox: np.ndarray,
    color: Tuple[int, int, int] = (0, 255, 0),
    thickness: int = 2,
    label: Optional[str] = None,
) -> np.ndarray:
    """
    Draw a bounding box on an RGB image.

    Args:
        image: input RGB image
        bbox: bounding box in (x, y, w, h) format
        color: RGB color
        thickness: line thickness
        label: optional label text

    Returns:
        image_with_box
    """
    # TODO: Implement this function
    raise NotImplementedError


def show_image(image: np.ndarray, title: str = "", figsize: Tuple[int, int] = (6, 4)):
    """Display an RGB image using matplotlib."""
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.axis("off")
    plt.title(title)
    plt.show()

## 1.6 Sequence Listing

This helper is useful for checking which sequences exist under `DATA_ROOT`.

In [ ]:
def list_sequences(data_root: Path) -> List[str]:
    """
    List valid sequence folders inside data_root.
    """
    # TODO: Implement this function
    raise NotImplementedError

## 1.7 Visualization Task

Visualize at least five frames from at least two training/validation sequences.

For each visualized frame, draw the ground-truth bounding box.

In [ ]:
# TODO: Visualize sample frames from full-GT sequences.

# Split Handling

The assignment uses fixed train/validation/test splits.

Use only the provided split files:

```text
splits/train.txt
splits/val.txt
splits/test.txt
```

Each file contains one sequence name per line.

## Split File Loading

Implement the functions below to read split files and verify that the sequence folders match the expected format.

In [ ]:
def read_split_file(split_path: Path) -> List[str]:
    """Read sequence names from a split file."""
    # TODO: Implement this function
    raise NotImplementedError


def load_splits(split_root: Path) -> Tuple[List[str], List[str], List[str]]:
    """Load train, validation, and test split files."""
    # TODO: Implement this function
    raise NotImplementedError


def verify_split_files(
    data_root: Path,
    train_sequences: List[str],
    val_sequences: List[str],
    test_sequences: List[str],
):
    """
    Verify that train/val sequences have groundtruth_rect.txt
    and test sequences have init_rect.txt.
    """
    # TODO: Implement this function
    raise NotImplementedError

## Load Splits

Run this cell after implementing the split-loading functions.

In [ ]:
# TODO: Load and verify splits

# (Optional) Dataset Exploration

This section is not required for implementation, but it is helpful for understanding the dataset.

You may use these cells to inspect:

- sequence lengths,
- bounding-box size distribution,
- object trajectory,
- scale changes.

These visualizations can also help your discussions.

## (Optional) Dataset Summary

After loading the splits, summarize the number of frames and bounding-box sizes in the training and validation sets.

In [ ]:
# TODO: Optional dataset summary.

## (Optional) Bounding-Box Distribution

Plot distributions of bounding-box width, height, area, or aspect ratio.

In [ ]:
# TODO: Optional bounding-box distribution plots.

# Example ideas:
# - histogram of bbox widths
# - histogram of bbox heights
# - histogram of bbox areas
# - histogram of aspect ratios

## (Optional) Target Motion Visualization

For one selected sequence, plot the target center coordinates over time.

In [ ]:
# TODO: Optional trajectory visualization.

# PART 2 — Search-Region Sample Generation

In single-object tracking, the target is initialized in the first frame. For later frames, the tracker searches around the previous target location.

In this assignment, you will train a model that receives a local search crop and predicts the target box inside that crop.

For a pair of consecutive frames:

```text
Frame t-1 bbox -> used to define the search region in frame t
Frame t bbox   -> target label inside the search region
```

## 2.1 Crop Metadata

`CropInfo` stores the information needed to convert between:

- full-frame coordinates,
- crop-local normalized coordinates.

In [ ]:
@dataclass
class CropInfo:
    """
    Metadata for mapping between crop coordinates and full-frame coordinates.

    x1, y1:
        Top-left location of the crop in the original image coordinate system.
        These may be negative if padding is used.

    crop_size:
        Side length of the square crop in original image pixels.

    output_size:
        Side length after resizing the crop for the CNN.
    """
    x1: float
    y1: float
    crop_size: float
    output_size: int

## 2.2 Bounding-Box Helper Functions

Implement simple helper functions for bounding-box center computation and clipping predictions to image boundaries.

In [ ]:
def bbox_center(bbox: np.ndarray) -> Tuple[float, float]:
    """Compute the center of a bbox in (x, y, w, h) format."""
    # TODO: Implement this function
    raise NotImplementedError


def clip_bbox_to_image(bbox: np.ndarray, image_shape: Tuple[int, int, int]) -> np.ndarray:
    """Clip a bbox to image boundaries and keep width/height positive."""
    # TODO: Implement this function
    raise NotImplementedError

## 2.3 Search-Region Crop

Use a square search region centered around the previous bounding box.

The recommended crop size is:

```text
crop_size = scale × max(previous_width, previous_height)
```

A reasonable default value is `scale = 2.5`.

If the crop goes outside the image, handle the boundary carefully. You may use padding or clipping, but your coordinate conversion must remain correct.

In [ ]:
def crop_search_region(
    image: np.ndarray,
    previous_bbox: np.ndarray,
    scale: float = 2.5,
    output_size: int = IMAGE_SIZE,
    pad_value: Tuple[int, int, int] = (128, 128, 128),
) -> Tuple[np.ndarray, CropInfo]:
    """
    Crop a square search region around the previous bounding box.

    Returns:
        crop_resized: RGB crop resized to output_size x output_size
        crop_info: metadata for coordinate conversion
    """
    # TODO: Implement this function
    raise NotImplementedError

## 2.4 Coordinate Conversion

The model predicts boxes inside the crop, but the final tracking output must be in full-frame coordinates.

You need two conversions:

1. Full-frame bbox → crop-local normalized bbox  
2. Crop-local normalized bbox → full-frame bbox

In [ ]:
def bbox_to_crop_coordinates(bbox: np.ndarray, crop_info: CropInfo) -> np.ndarray:
    """
    Convert a full-frame bbox into normalized crop coordinates.

    Returns:
        normalized_bbox: (x, y, w, h), usually in [0, 1]
    """
    # TODO: Implement this function
    raise NotImplementedError


def crop_to_frame_coordinates(normalized_bbox: np.ndarray, crop_info: CropInfo) -> np.ndarray:
    """
    Convert a normalized crop-local bbox back to full-frame coordinates.
    """
    # TODO: Implement this function
    raise NotImplementedError

## 2.5 Search-Region Visualization

Visualize several generated search-region samples.

For each sample, draw the current target box inside the search crop.

This is one of the most important sanity checks in the assignment.

In [ ]:
# TODO: Visualize generated search-region samples.

# PART 3 — CNN-Based Bounding-Box Regression Model

In this part, you will train a CNN to predict the target bounding box inside a search crop.

The model input is:

```text
224 × 224 RGB search crop
```

The model output is:

```text
x, y, width, height
```

The output must be crop-local and normalized to `[0, 1]`.

Do not train the model with full-frame coordinates.

## 3.1 Model Architecture Overview

Recommended architecture:

```text
Search Region Crop
        ↓
Preprocessing
        ↓
CNN Backbone
        ↓
Regression Head
        ↓
Normalized Bounding Box [x, y, w, h]
        ↓
Convert to full-frame coordinates during tracking
```

You may use an ImageNet-pretrained ResNet18 backbone. Replace its final classification layer with a regression layer that outputs four values.

## 3.2 PyTorch Dataset

Implement a PyTorch dataset that generates search-region samples from training sequences.

Each sample should contain:

```text
input: search crop from current frame
target: current target bbox in crop-local normalized coordinates
```

Use validation sequences similarly for crop-level validation.

In [ ]:
class SearchRegionDataset(Dataset):
    """
    Dataset that generates search-region samples from full-GT sequences.

    Each sample:
        input: current frame search crop around previous GT bbox
        target: current GT bbox in normalized crop coordinates
    """

    def __init__(
        self,
        data_root: Path,
        sequence_names: List[str],
        image_size: int = IMAGE_SIZE,
        search_scale: float = SEARCH_SCALE,
        augment: bool = False,
    ):
        self.data_root = Path(data_root)
        self.sequence_names = sequence_names
        self.image_size = image_size
        self.search_scale = search_scale
        self.augment = augment

        self.samples = []

        # TODO:
        # For each sequence:
        #   1. Load frame paths and bboxes using load_sequence_with_gt.
        #   2. For each t from 1 to len(sequence)-1:
        #        store current frame path,
        #        previous bbox,
        #        current bbox,
        #        sequence name,
        #        frame index.

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

        # Remove this after implementing the sample construction above.
        raise NotImplementedError

    def __len__(self):
        # TODO: Return number of samples
        raise NotImplementedError

    def __getitem__(self, idx):
        # TODO:
        # 1. Read current frame.
        # 2. Create search crop around previous bbox.
        # 3. Convert current bbox to crop-local normalized coordinates.
        # 4. Apply transform to crop.
        # 5. Return crop tensor and target tensor.
        raise NotImplementedError

## 3.3 DataLoaders

Create train and validation DataLoaders.

Training samples should come only from the training split. Validation samples should come only from the validation split.

In [ ]:
def create_dataloaders(
    data_root: Path,
    train_sequences: List[str],
    val_sequences: List[str],
    batch_size: int = BATCH_SIZE,
    image_size: int = IMAGE_SIZE,
    search_scale: float = SEARCH_SCALE,
):
    """Create train and validation dataloaders."""
    # TODO: Create train and validation datasets and dataloaders
    raise NotImplementedError


# TODO: Create dataloaders after loading splits

## 3.4 Model Definition

Define your CNN-based bounding-box regressor.

Recommended design:
- pretrained ResNet18 backbone,
- final regression layer with 4 outputs,
- output activation to keep predictions in `[0, 1]`.

In [ ]:
class TrackingRegressor(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()

        # TODO:
        # 1. Load ResNet18, optionally with pretrained weights.
        # 2. Replace the final classification layer with a regression layer.
        # 3. The regression layer should output 4 values.
        raise NotImplementedError

    def forward(self, x):
        # TODO:
        # Return normalized bbox prediction.
        # Hint: You may use sigmoid to keep predictions in [0, 1].
        raise NotImplementedError


def create_model() -> nn.Module:
    # TODO: Create and return model on DEVICE
    raise NotImplementedError


# TODO: Initialize model

## 3.5 IoU for Crop-Level Validation

Implement IoU calculation for boxes in `(x, y, w, h)` format.

This function will be used both for crop-level validation and sequence-level evaluation.

In [ ]:
def compute_iou_np(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """Compute IoU between two boxes in (x, y, w, h) format."""
    # TODO: Implement IoU calculation
    raise NotImplementedError


def mean_iou_batch(preds: torch.Tensor, targets: torch.Tensor) -> float:
    """Compute mean IoU for a batch of predicted and target boxes."""
    # TODO: Implement batch mean IoU
    raise NotImplementedError

## 3.6 Training and Validation Loops

Use a regression loss such as Smooth L1 loss.

During training, update model parameters.  
During validation, do not update model parameters.

Record:
- training loss,
- validation loss,
- validation crop-level mean IoU.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """Train model for one epoch."""
    # TODO: Implement training loop for one epoch
    raise NotImplementedError


@torch.no_grad()
def validate_model(model, loader, criterion, device):
    """
    Validate model.

    Returns:
        dictionary with validation loss and validation crop-level mean IoU
    """
    # TODO: Implement validation loop
    raise NotImplementedError


def train_model(
    model,
    train_loader,
    val_loader,
    num_epochs: int = NUM_EPOCHS,
    lr: float = LEARNING_RATE,
    device: torch.device = DEVICE,
):
    """Full training loop."""
    # TODO:
    # 1. Define loss function.
    # 2. Define optimizer.
    # 3. Train for num_epochs.
    # 4. Store train loss, validation loss, and validation crop-level mean IoU.
    # 5. Save or keep the best model according to validation performance.
    raise NotImplementedError

## 3.7 Train the Model

Train your model using the training split and monitor performance on the validation split.

Save the best model according to validation performance.

In [ ]:
# TODO: Train and save your model

## 3.8 Training Curves

Plot:
- training loss,
- validation loss,
- validation crop-level mean IoU.

Briefly comment on whether the model appears to learn.

In [ ]:
# TODO: Plot training and validation loss curves.
# TODO: Plot validation crop-level mean IoU.

## 3.9 Visualize Crop-Level Validation Predictions

Visualize at least five validation crop predictions.

Draw:
- ground-truth box,
- predicted box.

This helps check whether the model learned meaningful localization.

In [ ]:
def denormalize_tensor_image(tensor_img: torch.Tensor) -> np.ndarray:
    """Convert a normalized tensor image back to a uint8 RGB image for visualization."""
    # TODO: Implement inverse normalization
    raise NotImplementedError


@torch.no_grad()
def visualize_validation_predictions(model, dataset, num_samples: int = 5):
    """Visualize model predictions on validation crop samples."""
    # TODO:
    # 1. Select sample indices.
    # 2. Run model prediction.
    # 3. Convert normalized bboxes to 224x224 crop pixel coordinates.
    # 4. Draw GT and prediction.
    raise NotImplementedError


# TODO: Visualize validation predictions
# visualize_validation_predictions(model, val_dataset, num_samples=5)

# PART 4 — Single-Object Tracking on Validation and Test Sequences

After training the model, use it as a tracker.

For validation sequences:
- use the first line of `groundtruth_rect.txt` as initialization,
- predict all later frames,
- evaluate after tracking.

For test sequences:
- use `init_rect.txt` as initialization,
- predict all later frames,
- save CSV files,
- do not evaluate locally.

## 4.1 Evaluation-Time Preprocessing

Use the same preprocessing during tracking that you used during training.

Do not apply random augmentation during validation or test tracking.

In [ ]:
eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

## 4.2 Predict a Box Inside One Crop

This helper runs the model on one search crop and returns a normalized crop-local bounding box.

In [ ]:
@torch.no_grad()
def predict_bbox_in_crop(model, crop_rgb: np.ndarray, device: torch.device = DEVICE) -> np.ndarray:
    """Run the model on a crop and return normalized bbox prediction."""
    # TODO: Implement model inference for one crop
    raise NotImplementedError

## 4.3 Tracking Loop

Implement the full single-object tracking loop.

The function should receive only:

```text
model
frame paths
initial bbox
```

It must not receive the full ground-truth array.

In [ ]:
@torch.no_grad()
def track_sequence(
    model,
    frame_paths: List[Path],
    initial_bbox: np.ndarray,
    device: torch.device = DEVICE,
    scale: float = SEARCH_SCALE,
    image_size: int = IMAGE_SIZE,
) -> np.ndarray:
    """
    Track the target object through a sequence.

    Args:
        model: trained tracking model
        frame_paths: sorted list of frame paths
        initial_bbox: first-frame bbox, from GT or init_rect.txt
        device: torch device
        scale: search-region scale

    Returns:
        predictions: numpy array of shape (N, 4), full-frame bboxes

    Important:
        This function must not use ground-truth boxes after frame 1.
    """
    # TODO:
    # 1. Initialize predictions with initial_bbox.
    # 2. For each next frame:
    #      create search crop around previous prediction,
    #      predict bbox inside crop,
    #      convert to full-frame coordinates,
    #      clip to image boundaries,
    #      save prediction.
    raise NotImplementedError

## 4.4 Save Prediction CSV Files

Save predictions in the required format.

The saved coordinates must be full-frame pixel coordinates.

In [ ]:
def save_predictions_csv(predictions: np.ndarray, output_path: Path):
    """
    Save predictions in CSV format:

    frame,x,y,w,h
    1,...
    2,...

    Coordinates must be full-frame pixel coordinates, not normalized crop coordinates.
    """
    # TODO: Implement this function
    raise NotImplementedError

## 4.5 Run Tracking on Validation Sequences

Use the first validation ground-truth box for initialization.

Save validation predictions under:

```text
outputs/validation/
```

In [ ]:
def run_tracking_on_validation_sequences(
    model,
    data_root: Path,
    sequence_names: List[str],
    output_root: Path,
    device: torch.device = DEVICE,
    scale: float = SEARCH_SCALE,
) -> Dict[str, np.ndarray]:
    """Track validation sequences and save CSV prediction files."""
    # TODO:
    # For each validation sequence:
    #   load frames and GT using load_sequence_with_gt
    #   use gt_bboxes[0] as initial_bbox
    #   run track_sequence
    #   save predictions
    raise NotImplementedError

## 4.6 Run Tracking on Test Sequences

Use `init_rect.txt` for initialization.

Save test predictions under:

```text
outputs/test/
```

These files will be submitted for evaluation.

In [ ]:
def run_tracking_on_test_sequences(
    model,
    data_root: Path,
    sequence_names: List[str],
    output_root: Path,
    device: torch.device = DEVICE,
    scale: float = SEARCH_SCALE,
) -> Dict[str, np.ndarray]:
    """Track test sequences and save CSV prediction files."""
    # TODO:
    # For each test sequence:
    #   load frames and initial bbox using load_test_sequence
    #   run track_sequence
    #   save predictions
    raise NotImplementedError

## 4.7 Execute Tracking

Run tracking on validation and test sequences.

Make sure the generated test CSV files are complete and correctly formatted.

In [ ]:
# TODO: Run tracking on validation sequences


# TODO: Run tracking on hidden test sequences and save CSV files


# Required Test Output Format

For each test sequence, save one CSV file:

```text
outputs/test/<sequence_name>_predictions.csv
```

Each CSV file must contain:

```text
frame,x,y,w,h
```

Rules:

1. Number of rows must equal the number of frames.
2. Frame numbering must start from 1.
3. The first-frame prediction should be the initialization box from `init_rect.txt`.
4. Coordinates must be full-frame pixel coordinates.
5. Do not save normalized crop coordinates.
6. Bounding boxes must be in `(x, y, width, height)` format.

# PART 5 — Validation Evaluation and Analysis

Evaluate your tracker on validation sequences using the provided validation ground truth.

Do not evaluate test sequences. Test ground truth is hidden and will be evaluated by the instructors.

Required validation metrics:

1. Mean IoU  
2. Success Rate @ IoU ≥ 0.5  
3. Average Center Error  
4. Failure Count, where failure means IoU < 0.1

## 5.1 Metric Functions

Implement IoU, center error, and sequence-level evaluation.

In [ ]:
def compute_iou(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """Compute intersection-over-union between two boxes in (x, y, w, h) format."""
    # TODO: You may call compute_iou_np or implement again
    raise NotImplementedError


def compute_center_error(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """Compute Euclidean distance between the centers of two boxes."""
    # TODO: Implement this function
    raise NotImplementedError


def evaluate_sequence(
    predictions: np.ndarray,
    ground_truth: np.ndarray,
    success_iou_threshold: float = 0.5,
    failure_iou_threshold: float = 0.1,
) -> Dict[str, float]:
    """
    Evaluate one validation tracking sequence.

    Returns:
        metrics dictionary with:
        - mean_iou
        - success_rate_0.5
        - avg_center_error
        - failure_count
        - num_frames
    """
    # TODO: Implement evaluation
    raise NotImplementedError

## 5.2 Evaluate All Validation Sequences

Create a result table with one row per validation sequence and one final average row.

In [ ]:
def evaluate_validation_sequences(
    predictions_by_sequence: Dict[str, np.ndarray],
    data_root: Path,
) -> pd.DataFrame:
    """Evaluate all validation sequences and return a result table."""
    # TODO: Implement evaluation over validation sequences
    raise NotImplementedError


# TODO: Evaluate validation predictions

## 5.3 Visualize Validation Tracking Results

Visualize tracking results for at least three validation sequences.

For each selected sequence, show:
- an early frame,
- a middle frame,
- a late frame.

Draw both the ground-truth and predicted bounding boxes.

In [ ]:
def visualize_validation_tracking_result(
    sequence_name: str,
    predictions: np.ndarray,
    data_root: Path,
    frame_indices: Optional[List[int]] = None,
):
    """Visualize tracking predictions and ground truth for selected validation frames."""
    # TODO: Implement visualization
    raise NotImplementedError


# TODO: Visualize tracking results for at least three validation sequences

## 5.4 Failure-Case Analysis

Find frames where the tracker performs poorly.

At minimum, show:
- three successful frames,
- three failure cases.

For each failure case, explain what went wrong.

In [ ]:
def find_worst_validation_frames(
    sequence_name: str,
    predictions: np.ndarray,
    data_root: Path,
    k: int = 3,
) -> List[int]:
    """Return indices of the k validation frames with the lowest IoU."""
    # TODO: Implement worst-frame mining
    raise NotImplementedError


# TODO: Visualize worst frames for selected validation sequences

# Optional — Tracking Video Export

You may create MP4 visualizations of your validation tracking results.

This is optional, but it can help you inspect long sequences more easily.

In [ ]:
def create_validation_tracking_video(
    sequence_name: str,
    predictions: np.ndarray,
    data_root: Path,
    output_path: Path,
    fps: int = 20,
):
    """Create an MP4 video showing ground-truth and predicted boxes for a validation sequence."""
    # TODO: Optional implementation
    raise NotImplementedError

# Discussion Questions

Write your discussions about the performance, best/failure cases, metrics, etc.

Use your validation metrics, visualizations, and failure cases as evidence.

#TODO Write here

# Final Implementation Summary

Before submission, summarize your final settings here.

## Final Settings

**Backbone architecture:** TODO  
**Pretrained weights used?** TODO  
**Input size:** TODO  
**Search-region scale:** TODO  
**Batch size:** TODO  
**Learning rate:** TODO  
**Number of epochs:** TODO  
**Optimizer:** TODO  
**Loss function:** TODO  

# Submission Checklist

Before submitting, make sure that:

- [ ] Your notebook runs from beginning to end without manual intervention.
- [ ] Your codes are written clear with proper descriptions.
- [ ] You use only the provided train/validation/test splits.
- [ ] You do not use ground-truth boxes after the first frame during validation tracking.
- [ ] Your validation result table is included in the notebook.
- [ ] Your visualizations and failure analysis are included in the notebook.
- [ ] Your test prediction CSV files are saved under `outputs/test/`.
- [ ] Your CSV files contain full-frame pixel coordinates, not normalized crop coordinates.
- [ ] Your discussions are completed.
- [ ] You included the checkpoint (model weights) to the `checkpoint/` folder.